# NB03: Recompute deltaG via dGPredictor with OPAM2 pKas

After NB02 updated ModelSEEDDatabase compound TSVs with OPAM2 pKa values,
this notebook rebuilds the dGPredictor compound cache and recomputes deltaG
for all feasible ModelSEED reactions.

**Key insight**: Only the Legendre transform (pH/ionic-strength correction)
changes. The group contribution fingerprints and trained BayesianRidge model
weights are unchanged. We rebuild the compound cache with new pKas, then
re-run prediction to get updated `dG_mean` values.

The Legendre transform formula:
```
dG0s = -cumsum([0] + pKas) * R * T * ln(10)
```
A 1 pKa unit change shifts dG by ~5.71 kJ/mol per microspecies.

In [ ]:
import sys
import os
import json
import gzip
import glob
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DG_ROOT = Path('/tmp/dGPredictor')
MSDB_ROOT = Path('/tmp/ModelSEEDDatabase')
PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
FIGURES_DIR = PROJECT_ROOT / 'figures'

sys.path.insert(0, str(DG_ROOT))
sys.path.insert(0, str(DG_ROOT / 'CC'))

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

## 1. Save existing (Marvin-based) predictions for comparison

Load the current compound cache and run predictions with old pKas first,
so we have a baseline to compare against.

In [ ]:
from dg_prediction_modelseed import ModelSEEDdGPredictor
from predict_all_modelseed import predict_all_batch

old_cache_path = DG_ROOT / 'CC' / 'data_cc' / 'modelseed_compounds.json.gz'

# Check if we already have old predictions saved
old_pred_path = DATA_DIR / 'dg_predictions_marvin.json'
if old_pred_path.exists():
    with open(old_pred_path) as f:
        old_results = json.load(f)
    print(f'Loaded {len(old_results):,} existing Marvin-based predictions')
else:
    print('Running predictions with current (Marvin) compound cache...')
    predictor_old = ModelSEEDdGPredictor()
    old_results = predict_all_batch(predictor_old, pH=7.0, I=0.25, T=298.15)
    with open(old_pred_path, 'w') as f:
        json.dump(old_results, f, indent=2)
    print(f'Saved {len(old_results):,} Marvin-based predictions to {old_pred_path}')

## 2. Rebuild compound cache with OPAM2 pKas

Re-read the updated ModelSEED compound TSVs (from NB02) and rebuild
the compound cache using `build_compound_cache()` from `retrain_modelseed.py`.

In [ ]:
from retrain_modelseed import (
    build_compound_cache,
    parse_modelseed_pka,
    parse_formula_to_atom_bag,
)

# Load updated compound TSVs
cpd_files = sorted(glob.glob(str(MSDB_ROOT / 'Biochemistry' / 'compound_*.tsv')))
cpd_files = [f for f in cpd_files if 'provenance' not in f]

compounds_df = pd.concat([pd.read_csv(f, sep='\t') for f in cpd_files], ignore_index=True)
print(f'Loaded {len(compounds_df):,} compounds from ModelSEEDDatabase')

# Load InChI map
inchi_path = DG_ROOT / 'CC' / 'data_cc' / 'modelseed_cpd_inchi.json'
if inchi_path.exists():
    with open(inchi_path) as f:
        cpd_inchi_map = json.load(f)
else:
    cpd_inchi_map = {}
    print('No InChI map found — using empty map')

print(f'InChI map entries: {len(cpd_inchi_map):,}')

In [ ]:
# Build new compound cache with OPAM2 pKas
new_compound_dict = build_compound_cache(compounds_df, cpd_inchi_map)
print(f'Built {len(new_compound_dict):,} compound objects')

# Save the new cache
new_cache_path = DG_ROOT / 'CC' / 'data_cc' / 'modelseed_compounds_opam2.json.gz'
cache_list = [comp.to_json_dict() for comp in new_compound_dict.values()]
with gzip.open(new_cache_path, 'wt', encoding='utf-8') as f:
    json.dump(cache_list, f)
print(f'Saved new compound cache to {new_cache_path}')

## 3. Run predictions with OPAM2 compound cache

In [ ]:
# Create predictor with new compound cache
predictor_new = ModelSEEDdGPredictor(
    compound_cache_path=str(new_cache_path)
)

new_results = predict_all_batch(predictor_new, pH=7.0, I=0.25, T=298.15)
print(f'Predicted dG for {len(new_results):,} reactions with OPAM2 pKas')

# Save
new_pred_path = DATA_DIR / 'dg_predictions_opam2.json'
with open(new_pred_path, 'w') as f:
    json.dump(new_results, f, indent=2)
print(f'Saved to {new_pred_path}')

## 4. Compare old vs new deltaG values

In [ ]:
common_rxns = set(old_results.keys()) & set(new_results.keys())
print(f'Reactions in both: {len(common_rxns):,}')

rows = []
for rxn_id in sorted(common_rxns):
    old_dg = old_results[rxn_id]['dG_mean']
    new_dg = new_results[rxn_id]['dG_mean']
    old_model = old_results[rxn_id]['dG_model_only']
    new_model = new_results[rxn_id]['dG_model_only']
    old_corr = old_results[rxn_id]['ddG0_pH_correction']
    new_corr = new_results[rxn_id]['ddG0_pH_correction']
    rows.append({
        'rxn_id': rxn_id,
        'old_dg': old_dg,
        'new_dg': new_dg,
        'abs_diff': abs(new_dg - old_dg),
        'diff': new_dg - old_dg,
        'old_model_only': old_model,
        'new_model_only': new_model,
        'old_pH_correction': old_corr,
        'new_pH_correction': new_corr,
        'pH_correction_diff': new_corr - old_corr,
    })

df_cmp = pd.DataFrame(rows)
df_cmp.to_csv(DATA_DIR / 'dg_comparison.tsv', sep='\t', index=False)
print(f'Saved {len(df_cmp):,} reaction comparisons to data/dg_comparison.tsv')

In [ ]:
print(f'deltaG change statistics:')
print(f'  Mean |diff|:   {df_cmp["abs_diff"].mean():.2f} kJ/mol')
print(f'  Median |diff|: {df_cmp["abs_diff"].median():.2f} kJ/mol')
print(f'  Max |diff|:    {df_cmp["abs_diff"].max():.2f} kJ/mol')
print(f'  Std of diff:   {df_cmp["diff"].std():.2f} kJ/mol')
print()
print(f'  Reactions with |diff| > 1 kJ/mol:  {(df_cmp["abs_diff"] > 1).sum():,} ({(df_cmp["abs_diff"] > 1).mean():.1%})')
print(f'  Reactions with |diff| > 5 kJ/mol:  {(df_cmp["abs_diff"] > 5).sum():,} ({(df_cmp["abs_diff"] > 5).mean():.1%})')
print(f'  Reactions with |diff| > 10 kJ/mol: {(df_cmp["abs_diff"] > 10).sum():,} ({(df_cmp["abs_diff"] > 10).mean():.1%})')
print()
print('Verification: model-only dG should be identical (group fingerprints unchanged):')
model_only_diff = (df_cmp['new_model_only'] - df_cmp['old_model_only']).abs()
print(f'  Max |model_only diff|: {model_only_diff.max():.6f} kJ/mol (should be ~0)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram of deltaG changes
axes[0].hist(df_cmp['diff'], bins=100, edgecolor='none', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('New dG - Old dG (kJ/mol)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'deltaG change distribution (n={len(df_cmp):,})')

# Parity plot: old vs new dG
axes[1].scatter(df_cmp['old_dg'], df_cmp['new_dg'], s=2, alpha=0.2)
lo = min(df_cmp['old_dg'].min(), df_cmp['new_dg'].min()) - 10
hi = max(df_cmp['old_dg'].max(), df_cmp['new_dg'].max()) + 10
axes[1].plot([lo, hi], [lo, hi], 'k--', lw=1)
axes[1].set_xlabel('Old dG (kJ/mol, Marvin pKa)')
axes[1].set_ylabel('New dG (kJ/mol, OPAM2 pKa)')
axes[1].set_title('Parity: old vs new deltaG')

# pH correction change
axes[2].hist(df_cmp['pH_correction_diff'], bins=100, edgecolor='none', alpha=0.7)
axes[2].axvline(0, color='red', linestyle='--', linewidth=1)
axes[2].set_xlabel('pH correction change (kJ/mol)')
axes[2].set_ylabel('Count')
axes[2].set_title('Legendre transform correction change')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'dg_comparison.png', dpi=150)
plt.show()

## 5. Top reactions by deltaG change

In [ ]:
print('Top 20 reactions by |deltaG change|:')
print(df_cmp.nlargest(20, 'abs_diff')[['rxn_id', 'old_dg', 'new_dg', 'diff', 'pH_correction_diff']].to_string(index=False))

## 6. Reactions that could change feasibility direction

A reaction that was thermodynamically favorable (dG < 0) but is now
unfavorable (dG > 0), or vice versa, represents a directionality flip.

In [ ]:
sign_changed = df_cmp[(df_cmp['old_dg'] * df_cmp['new_dg']) < 0]
print(f'Reactions with sign change in dG: {len(sign_changed):,} ({len(sign_changed)/len(df_cmp):.1%})')
print(f'  Old favorable -> New unfavorable: {((sign_changed["old_dg"] < 0) & (sign_changed["new_dg"] > 0)).sum()}')
print(f'  Old unfavorable -> New favorable: {((sign_changed["old_dg"] > 0) & (sign_changed["new_dg"] < 0)).sum()}')